# Imports and Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog novocart_catalog")

In [0]:
spark.sql("create schema if not exists novocart_catalog.silver_schema")

In [0]:
silver_run_id = str(uuid.uuid4())
print(f"Current Silver Run ID: {silver_run_id}")

# Silver Control Table

In [0]:
spark.sql("""create table if not exists novocart_catalog.silver_schema.processing_control(
    layer string,
    entity_name string,
    last_processed_bronze_run_id string,
    last_processed_bronze_ingested_at timestamp,
    rows_merged bigint,
    run_status string,
    silver_run_id string,
    updated_at timestamp
) using delta
""")

# Helper Functions

In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)

        (
            dt.alias("t")
            .merge(df_source.alias("s"), f"t.{join_key} = s.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl = (
        spark.read.table("novocart_catalog.silver_schema.processing_control")
        .filter(
            (col("layer") == "silver")
            & (col("entity_name") == entity_name)
            & (col("run_status") == "success")
        )
        .orderBy(col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None, None
    else:
        return rows[0]["last_processed_bronze_ingested_at"], rows[0]['last_processed_bronze_run_id']

In [0]:
def upsert_silver_control(
    entity_name,
    last_processed_bronze_run_id,
    last_processed_bronze_ingested_at,
    rows_merged,
):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "success",
                silver_run_id,
                datetime.utcnow(),
            )
        ],
        schema="layer string, entity_name string, last_processed_bronze_run_id string, last_processed_bronze_ingested_at timestamp, rows_merged bigint, run_status string, silver_run_id string, updated_at timestamp",
    )

    dt = DeltaTable.forName(spark, "novocart_catalog.silver_schema.processing_control")

    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"), "t.layer = s.layer AND t.entity_name = s.entity_name"
        )
        .whenMatchedUpdate(
            set={
                "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
                "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
                "rows_merged": "s.rows_merged",
                "run_status": "s.run_status",
                "silver_run_id": "s.silver_run_id",
                "updated_at": "s.updated_at",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at, last_run_id = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df, last_ingested_at, last_run_id
    else:
        return bronze_df.filter(col("bronze_ingested_at") > lit(last_ingested_at)), last_ingested_at, last_run_id

# Orders Incremental Processing

In [0]:
df_raw_order = spark.sql("select * from novocart_catalog.bronze_schema.orders_raw")
display(df_raw_order)

In [0]:
orders_inc, last_orders_ingested_at, last_order_run_id = get_incremental_bronze(
    "novocart_catalog.bronze_schema.orders_raw", "orders"
)

orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver: {orders_inc_count}")

if orders_inc_count > 0:
    order_window = Window.partitionBy("order_id").orderBy(
        col("updated_at").cast("timestamp").desc(), col("bronze_ingested_at").desc()
    )

    orders_cleaned = (
        orders_inc.withColumn("order_status", upper(trim(col("order_status"))))
        .withColumn(
            "order_status",
            when(col("order_status") == "", lit(None)).otherwise(col("order_status")),
        )
        .withColumn("order_amount", regexp_replace(col("order_amount"), r"[$, ]", ""))
        .withColumn(
            "order_amount",
            when(
                trim(col("order_amount")).isin("N/A", "NULL", "??", ""), None
            ).otherwise(col("order_amount")),
        )
        .withColumn("order_amount", col("order_amount").cast("double"))
        .withColumn("created_at", to_timestamp(col("created_at")))
        .withColumn("updated_at", to_timestamp(col("updated_at")))
        .withColumn("row_rank", row_number().over(order_window))
        .filter(col("row_rank") == 1)
        .drop(col("row_rank"))
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    upsert_to_silver(
        orders_cleaned, "novocart_catalog.silver_schema.orders_cleaned", "order_id"
    )

    orders_validated = (
        orders_cleaned.withColumn(
            "to_be_verified_by_order_team",
            when(col("customer_id").isNull(), "verify_customer_id")
            .when(col("product_id").isNull(), "verify_product_id")
            .when(
                (col("order_status").isNull() | (trim(col("order_status")) == "")),
                "verify_order_status",
            )
            .when(
                col("order_amount").isNull() | (col("order_amount") <= 0),
                "verify_order_amount",
            )
            .otherwise("No Issues"),
        )
        .withColumn(
            "check_order_amount",
            when(
                col("order_amount").isNull() | (col("order_amount") <= 0), lit(True)
            ).otherwise(False),
        )
        .withColumn("order_date", to_date(col("created_at")))
        .withColumn("order_year", year(col("created_at")))
        .withColumn("order_month", month(col("created_at")))
        .withColumn("order_day", dayofmonth(col("created_at")))
        .withColumn("order_dow", date_format(col("created_at"), "E"))
    )

    orders_good = orders_validated.filter(
        col("to_be_verified_by_order_team") == "No Issues"
    )

    orders_bad = orders_validated.filter(
        col("to_be_verified_by_order_team") != "No Issues"
    ).withColumn("quarantine_ts", current_timestamp())

    upsert_to_silver(
        orders_good, "novocart_catalog.silver_schema.orders_transformed", "order_id"
    )

    orders_bad.write.format("delta").mode("append").saveAsTable(
        "novocart_catalog.silver_schema.orders_quarantine"
    )

    max_ingested = orders_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0][
        "mx"
    ]

    mx_run = (
        orders_inc.filter(col("bronze_ingested_at") == lit(max_ingested))
        .agg(max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("orders", mx_run, max_ingested, orders_good.count())

else:
    print("No new orders Bronze rows for silver")

    upsert_silver_control("orders", last_order_run_id, last_orders_ingested_at, orders_inc_count)

In [0]:
%sql
select * from novocart_catalog.silver_schema.orders_quarantine

# Products Incremental Processing

In [0]:
products_inc, last_products_ingested_at, last_product_run_id = get_incremental_bronze(
    "novocart_catalog.bronze_schema.products_raw", "products"
)

products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver: {products_inc_count}")

if products_inc_count > 0:
    product_window = Window.partitionBy("product_id").orderBy(
        col("updated_at").cast("timestamp").desc(), col("bronze_ingested_at").desc()
    )

    products_cleaned = (
        products_inc.withColumn("product_name", upper(trim(col("product_name"))))
        .withColumn("product_name", regexp_replace(col("product_name"), r"[-_]"," "))
        .withColumn(
            "product_name",
            when(col("product_name") == "", lit(None)).otherwise(col("product_name")),
        )
        .withColumn(
            "category",
            when(
                upper(trim(col("category"))).contains("ELECTRNICS"), lit("ELECTRONICS")
            ).otherwise(upper(trim(col("category")))),
        )
        .withColumn("price", trim(col("price")))
        .withColumn("price", regexp_replace(col("price"), r"\$", ""))
        .withColumn("price", regexp_replace(col("price"), r",", "."))
        .withColumn("price", regexp_replace(col("price"), r"\s+", ""))
        .withColumn("price", expr("try_cast(price as double)"))
        .withColumn("updated_at", to_timestamp(col("updated_at")))
        .withColumn("row_rank", row_number().over(product_window))
        .filter(col("row_rank") == 1)
        .drop(col("row_rank"))
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    upsert_to_silver(
        products_cleaned,
        "novocart_catalog.silver_schema.products_cleaned",
        "product_id",
    )

    products_validated = products_cleaned.withColumn(
        "to_be_verified_by_product_team",
        when(col("product_name").isNull(), "verify_product_name")
        .when(col("category").isNull(), "verify_category")
        .when(
            col("price").isNull() | (col("price") <= 0),
            "verify_price",
        )
        .otherwise("No Issues"),
    ).withColumn(
        "check_product_price",
        when(
            col("price").isNull() | (col("price") <= 0), lit("invalid_price")
        ).otherwise("valid_price"),
    )

    products_good = products_validated.filter(
        (col("to_be_verified_by_product_team") == "No Issues")
        & (col("check_product_price") == "valid_price")
    )

    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")

    products_bad = products_validated.filter(
        (col("to_be_verified_by_product_team") != "No Issues")
        & (col("check_product_price") == "invalid_price")
    ).withColumn("quarantine_ts", current_timestamp())

    upsert_to_silver(
        products_good, "novocart_catalog.silver_schema.products_transformed", "product_id"
    )

    products_bad.write.format("delta").mode("append").saveAsTable(
        "novocart_catalog.silver_schema.products_quarantine"
    )

    max_ingested = products_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0][
        "mx"
    ]

    mx_run = (
        products_inc.filter(col("bronze_ingested_at") == lit(max_ingested))
        .agg(max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("products", mx_run, max_ingested, products_good.count())

else:
    print("No new products bronze rows for silver")

    upsert_silver_control("products", last_product_run_id, last_products_ingested_at, products_inc_count)

In [0]:
%sql
select * from novocart_catalog.silver_schema.products_quarantine

# Payments Incremental Processing

In [0]:
payments_inc, last_payments_ingested_at, last_payment_run_id = get_incremental_bronze(
    "novocart_catalog.bronze_schema.payments_raw", "payments"
)

payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver: {payments_inc_count}")

if payments_inc_count > 0:
    payment_window = Window.partitionBy("payment_id").orderBy(
        col("processed_at").cast("timestamp").desc(), col("bronze_ingested_at").desc()
    )

    payments_cleaned = (
        payments_inc.withColumn("payment_status", upper(trim(col("payment_status"))))
        .withColumn(
            "payment_status",
            when(col("payment_status") == "", lit(None)).otherwise(
                col("payment_status")
            ),
        )
        .withColumn("paid_amount", trim(col("paid_amount")))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), r"\$", ""))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), r",", "."))
        .withColumn("paid_amount", regexp_replace(col("paid_amount"), r"\s+", ""))
        .withColumn("paid_amount", expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", to_timestamp(col("processed_at")))
        .withColumn("row_rank", row_number().over(payment_window))
        .filter(col("row_rank") == 1)
        .drop(col("row_rank"))
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    upsert_to_silver(
        payments_cleaned,
        "novocart_catalog.silver_schema.payments_cleaned",
        "payment_id",
    )

    payments_validated = payments_cleaned.withColumn(
        "to_be_verified_by_payment_team",
        when(col("order_id").isNull(), "verify_order_id")
        .when(col("payment_status").isNull(), "verify_payment_status")
        .when(
            col("paid_amount").isNull() | (col("paid_amount") <= 0),
            "verify_paid_amount",
        )
        .otherwise("No Issues"),
    ).withColumn(
        "check_paid_amount",
        when(
            col("paid_amount").isNull() | (col("paid_amount") <= 0), lit(True)
        ).otherwise(False),
    )

    payments_good = payments_validated.filter(
        (col("to_be_verified_by_payment_team") == "No Issues")
        & (col("check_paid_amount") == False)
    )

    payments_bad = payments_validated.filter(
        (col("to_be_verified_by_payment_team") != "No Issues")
        & (col("check_paid_amount") == True)
    ).withColumn("quarantine_ts", current_timestamp())

    upsert_to_silver(
        payments_good,
        "novocart_catalog.silver_schema.payments_transformed",
        "payment_id",
    )

    payments_bad.write.format("delta").mode("append").saveAsTable(
        "novocart_catalog.silver_schema.payments_quarantine"
    )

    max_ingested = payments_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0][
        "mx"
    ]

    mx_run = (
        payments_inc.filter(col("bronze_ingested_at") == lit(max_ingested))
        .agg(max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("payments", mx_run, max_ingested, payments_good.count())

else:
    print("No new payments bronze rows for silver")

    upsert_silver_control(
        "payments", last_payment_run_id, last_products_ingested_at, products_inc_count
    )

In [0]:
print("Orders Silver Count: ", spark.sql("select count(*) from novocart_catalog.silver_schema.orders_transformed").collect()[0][0])
print("Products Silver Count: ", spark.sql("select count(*) from novocart_catalog.silver_schema.products_transformed").collect()[0][0])
print("Payments Silver Count: ", spark.sql("select count(*) from novocart_catalog.silver_schema.payments_transformed").collect()[0][0])

display(spark.sql("select * from novocart_catalog.silver_schema.processing_control").orderBy(col("entity_name")))